# NeuroCore: Brain-Inspired Coreset Selection for Lightweight VLA

> Course project: Brain-inspired coreset selection for lightweight Vision-Language-Action (VLA) robotic arm action prediction.

**Dataset**: ALOHA Sim Transfer Cube (Human Demonstrations) — 50 episodes, ~20 000 frames  
**Task**: Predict single-arm 7-DoF actions from frozen ResNet-18 visual features  
**Environment**: Local (Jupyter) or Google Colab

## Colab Setup (run first if on Google Colab)

If you are running this notebook on **Google Colab**, execute the cell below to clone the repository and install dependencies. If running locally, you can skip this cell.

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Running in Google Colab — setting up environment...')
    if not os.path.exists('NeuroCore'):
        !git clone https://github.com/YOUR_USERNAME/NeuroCore.git
    %cd NeuroCore
    !pip install -q -r requirements.txt
    print('Setup complete. Working directory:', os.getcwd())
else:
    print('Running locally — skipping Colab setup.')
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
        print('Changed to project root:', os.getcwd())


## 1. Imports & Configuration

In [ ]:
import os
import sys
import json

project_root = os.path.abspath('.') if os.path.exists('src') else os.path.abspath('..')
sys.path.insert(0, project_root)

import numpy as np
import torch
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

RESULTS_DIR = os.path.join(project_root, 'results')
FEATURE_PATH = os.path.join(RESULTS_DIR, 'features_resnet18.pt')
BASELINE_METRICS_PATH = os.path.join(RESULTS_DIR, 'baseline_metrics.json')
CORESET_PATH = os.path.join(RESULTS_DIR, 'coreset_selection.json')
CORESET_METRICS_PATH = os.path.join(RESULTS_DIR, 'coreset', 'coreset_metrics.json')

RUN_TRAINING = True
EXTRACT_FEATURES = False

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print('CUDA available:', torch.cuda.is_available())
print('Project root:', project_root)
print('Feature path:', FEATURE_PATH)
print('Feature file exists:', os.path.exists(FEATURE_PATH))


## 2. Feature Extraction (one-time)

Visual features are extracted **offline** using a frozen ResNet-18. Features are cached to disk so extraction is a one-time cost.

> **Note**: If you cloned this repository, the cached features should already be present. If not, set `EXTRACT_FEATURES = True` below (requires LeRobot dataset with images).

In [ ]:
if EXTRACT_FEATURES or not os.path.exists(FEATURE_PATH):
    print('Extracting ResNet-18 features...')
    from src.feature_extractor import extract_and_cache_features
    features_dict = extract_and_cache_features(FEATURE_PATH)
else:
    print('Loading cached features...')
    features_dict = torch.load(FEATURE_PATH, weights_only=True)
    features_dict = {
        (int(k[0]), int(k[1])): v
        for k, v in features_dict.items()
    }
print(f'Loaded {len(features_dict)} features, dim={list(features_dict.values())[0].shape[0]}')


## 3. Dataset

We use the **ALOHA Sim Transfer Cube (Human Demonstrations)** dataset from Hugging Face LeRobot.

- **Episodes**: 50 successful human demonstrations
- **Frames**: ~20 000 total (400 frames per episode)
- **Actions**: 14-DoF joint actions (we use the first 7 dimensions)

For this project, we reduce to a single camera view and single-arm 7-DoF actions.

In [ ]:
from src.data_utils import load_aloha_dataset

dataset = load_aloha_dataset()
num_frames = len(dataset)
episodes = [int(item['episode_index']) for item in dataset]
num_episodes = max(episodes) + 1
frames_per_episode = num_frames // num_episodes

print(f'Total frames: {num_frames}')
print(f'Total episodes: {num_episodes}')
print(f'Frames per episode: {frames_per_episode}')
print(f'Action dimensions: {len(dataset[0]["action"])} (using first 7 for single arm)')

missing = 0
for i in range(len(dataset)):
    item = dataset[i]
    key = (int(item['episode_index']), int(item['frame_index']))
    if key not in features_dict:
        missing += 1
print(f'Frames without cached features: {missing}')


## 4. Baseline — Random 10% Sampling

Our baseline randomly selects 5 episodes (10% of 50) and trains a lightweight MLP:

- **Architecture**: 512 → 256 → 128 → 7
- **Loss**: Mean Squared Error (MSE)
- **Optimizer**: Adam (lr = 1e-3)
- **Epochs**: 50  |  **Batch size**: 32

The model is evaluated on the remaining 45 episodes.

In [ ]:
from src.baseline import run_baseline, BaselineConfig

if RUN_TRAINING or not os.path.exists(BASELINE_METRICS_PATH):
    print('Training baseline MLP on random 5 episodes...')
    baseline_metrics = run_baseline(
        feature_path=FEATURE_PATH,
        config=BaselineConfig(seed=SEED),
        save_dir=RESULTS_DIR,
    )
    print(f'Baseline MSE: {baseline_metrics["mse"]:.6f}')
else:
    print('Loading cached baseline metrics...')
    with open(BASELINE_METRICS_PATH, 'r') as f:
        baseline_metrics = json.load(f)
    print(f'Baseline MSE: {baseline_metrics["mse"]:.6f}')

print(f'Train episodes: {baseline_metrics["train_episodes"]}')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(baseline_metrics['train_losses'], label='Train', color='blue')
ax1.plot(baseline_metrics['val_losses'], label='Validation', color='orange')
ax1.set_title('Baseline Loss Curves')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE')
ax1.legend()
ax1.grid(True, alpha=0.3)

joints = list(range(7))
ax2.bar(joints, baseline_metrics['per_joint_mse'], color='steelblue', edgecolor='black')
ax2.set_title('Baseline Per-Joint Test MSE')
ax2.set_xlabel('Joint Index')
ax2.set_ylabel('MSE')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Coreset Selection Algorithm

We replace random sampling with a brain-inspired pruning algorithm that fuses two filters:

### 5.1 Temporal Filter — Predictive Coding
For each episode, compute the mean frame-to-frame action change:
$$\delta_t = \|a_t - a_{t-1}\|_2$$
Episodes with high motion complexity score higher.

### 5.2 Distributional Filter — RAS
Cluster all frame-level ResNet-18 features (K-Means, k=15). For each episode, compute entropy of cluster occupancy × coverage ratio.

### 5.3 Unified Selection
Normalize both scores to [0,1] and fuse with weight $\alpha$:
$$S_{\text{final}} = \alpha \cdot \tilde{S}_{\text{temp}} + (1 - \alpha) \cdot \tilde{S}_{\text{dist}}$$

After ablation, we use **$\alpha = 0.75$** (temporal-heavy, diversity-tempered).

In [ ]:
from src.coreset.select import select_coreset

if RUN_TRAINING or not os.path.exists(CORESET_PATH):
    print('Running coreset selection (alpha=0.75)...')
    selected_episodes, final_scores, temp_norm, dist_norm = select_coreset(
        dataset, features_dict, k=5, alpha=0.75, save_path=CORESET_PATH
    )
    print(f'Selected episodes: {selected_episodes}')
else:
    print('Loading cached coreset selection...')
    with open(CORESET_PATH, 'r') as f:
        coreset_data = json.load(f)
    selected_episodes = coreset_data['selected_episodes']
    final_scores = {int(k): v for k, v in coreset_data['scores'].items()}
    temp_norm = {int(k): v for k, v in coreset_data['temporal_scores'].items()}
    dist_norm = {int(k): v for k, v in coreset_data['distributional_scores'].items()}
    print(f'Selected episodes: {selected_episodes}')


In [ ]:
episodes = sorted(list(final_scores.keys()))
t_vals = [temp_norm[e] for e in episodes]
d_vals = [dist_norm.get(e, 0) for e in episodes]
f_vals = [final_scores[e] for e in episodes]
colors = ['red' if e in selected_episodes else 'gray' for e in episodes]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(episodes, t_vals, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('Temporal Score (Predictive Coding)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Normalized Score')
axes[0].grid(True, alpha=0.3)

axes[1].bar(episodes, d_vals, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_title('Distributional Score (RAS)')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Normalized Score')
axes[1].grid(True, alpha=0.3)

axes[2].bar(episodes, f_vals, color=colors, edgecolor='black', alpha=0.8)
axes[2].set_title('Combined Score (alpha=0.75) — Top-5 in Red')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Normalized Score')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Validation — Coreset vs. Baseline

We retrain the **identical MLP architecture** on the selected coreset episodes and evaluate on the **same held-out 45 episodes** as the baseline. This isolates the effect of data selection from model architecture.

In [ ]:
from src.validate import run_validation

if RUN_TRAINING or not os.path.exists(CORESET_METRICS_PATH):
    print('Running coreset validation...')
    validation_results = run_validation(
        coreset_path=CORESET_PATH,
        feature_path=FEATURE_PATH,
        baseline_metrics_path=BASELINE_METRICS_PATH,
        save_dir=os.path.join(RESULTS_DIR, 'coreset'),
    )
    coreset_metrics = validation_results['coreset_metrics']
else:
    print('Loading cached validation results...')
    with open(CORESET_METRICS_PATH, 'r') as f:
        comparison = json.load(f)
    with open(os.path.join(RESULTS_DIR, 'coreset', 'baseline_metrics.json'), 'r') as f:
        coreset_metrics = json.load(f)
    print(f'Coreset MSE: {coreset_metrics["mse"]:.6f}')
    print(f'Baseline MSE: {comparison["baseline_mse"]:.6f}')
    print(f'Improvement: {comparison["improvement_percent"]:.2f}%')


In [ ]:
with open(BASELINE_METRICS_PATH, 'r') as f:
    baseline = json.load(f)

with open(os.path.join(RESULTS_DIR, 'coreset', 'baseline_metrics.json'), 'r') as f:
    coreset = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(baseline['val_losses'], label='Random Baseline', color='gray', linestyle='--', linewidth=2)
ax1.plot(coreset['val_losses'], label='Coreset', color='blue', linewidth=2)
ax1.set_title('Validation Loss Convergence')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE')
ax1.legend()
ax1.grid(True, alpha=0.3)

joints = list(range(7))
width = 0.35
ax2.bar([j - width/2 for j in joints], baseline['per_joint_mse'], width,
        label='Random Baseline', color='gray', edgecolor='black')
ax2.bar([j + width/2 for j in joints], coreset['per_joint_mse'], width,
        label='Coreset', color='blue', edgecolor='black')
ax2.set_title('Per-Joint Test MSE')
ax2.set_xlabel('Joint Index')
ax2.set_ylabel('MSE')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n' + '='*50)
print(f"{'Method':<20} {'Test MSE':<15} {'Improvement'}")
print('='*50)
print(f"{'Random Baseline':<20} {baseline['mse']:.6f}     —")
improvement = (baseline['mse'] - coreset['mse']) / baseline['mse'] * 100
print(f"{'Coreset (Ours)':<20} {coreset['mse']:.6f}     +{improvement:.2f}%")
print('='*50)


## 7. Discussion & Conclusion

### Why does the coreset work?

The optimal $\alpha = 0.75$ suggests that **temporal action novelty** is the dominant signal for learning, but must be tempered with **visual diversity** to avoid over-selecting chaotic episodes. This mirrors neuroscience: predictive coding drives cortical learning, but the RAS modulates which errors reach conscious processing.

### Limitations

1. **Dataset size**: 50 episodes is small; random baseline variance is high.
2. **Episode-level granularity**: Frame-level pruning could yield further gains.
3. **Single view, single arm**: Multi-modal fusion is left for future work.

### Definition of Redundancy

We define redundancy at two timescales:

1. **Temporal redundancy**: A frame is redundant if $\delta_t = \|a_t - a_{t-1}\|_2$ is small — the action is predictable from the previous frame.
2. **Distributional redundancy**: A frame is redundant if it belongs to a visual cluster already well-represented in the selected set.

This definition is brain-inspired because it explicitly discards data that the brain's own filtering mechanisms (predictive coding + RAS) would "explain away."

### Reproduction

```bash
python -m src.baseline        # Run random baseline
python -m src.coreset.select  # Run coreset selection
python -m src.validate        # Validate and compare
```